In [9]:
import os
from sqlalchemy import create_engine
from jinja2 import Environment, FileSystemLoader
import pandas as pd
import geopandas as gpd
import datetime
import shapely

In [10]:
TARGET_DATE = os.environ.get("TARGET_DATE", "2026-07-05")
TARGET_ROUTE_SHORT_NAME = os.environ.get("TARGET_ROUTE_SHORT_NAME", "ECR")
TARGET_SCHEDULE_FEED = os.environ.get("TARGET_SCHEDULE_FEED", "Bay Area 511 SamTrans Schedule")
CONSTANT_MACROS = {
    "TARGET_DATE": TARGET_DATE,
    "TARGET_ROUTE_SHORT_NAME": TARGET_ROUTE_SHORT_NAME,
    "TARGET_SCHEDULE_FEED": TARGET_SCHEDULE_FEED
}
CONSTANT_MACROS

{'TARGET_DATE': '2026-07-05',
 'TARGET_ROUTE_SHORT_NAME': 'ECR',
 'TARGET_SCHEDULE_FEED': 'Bay Area 511 SamTrans Schedule'}

In [11]:
# gcp boilerplate
CALITP_BQ_MAX_BYTES = os.environ.get("CALITP_BQ_MAX_BYTES", 5_000_000_000)
CALITP_BQ_LOCATION = os.environ.get("CALITP_BQ_LOCATION", "us-west2")
engine = create_engine(
    f"bigquery://cal-itp-data-infra/?maximum_bytes_billed={CALITP_BQ_MAX_BYTES}",  # noqa: E231
    location=CALITP_BQ_LOCATION
)
env = Environment(loader=FileSystemLoader("templates"))


In [12]:
def make_linestring(x: str) -> shapely.geometry.LineString:
    # shapely errors if the array contains only one point
    if len(x) > 1:
        # each point in the array is wkt
        # so convert them to shapely points via list comprehension
        as_wkt = [shapely.wkt.loads(i) for i in x]
        return shapely.geometry.LineString(as_wkt)
    return shapely.geometry.LineString()

# Get the target route and feed
matching_shapes_df = pd.read_sql_query(
    env.get_template("shapes_routes.sql").render(CONSTANT_MACROS),
    engine
)
matching_shapes_geom = gpd.GeoSeries(
    matching_shapes_df["pt_array"].map(make_linestring), crs=4326
)
matching_shapes = gpd.GeoDataFrame(
    matching_shapes_df.drop(["pt_array"], axis=1),
    geometry=matching_shapes_geom,
)
feed_key = matching_shapes.loc[0, "feed_key"]
matching_shapes

,shape_id,feed_key,gtfs_dataset_name,route_id,route_short_name,ct,geometry
0,ECR2069,d71492afe126cd61f629b17c3c056fcd,Bay Area 511 SamTrans Schedule,ECR,ECR,66,"LINESTRING (-122.16599 37.44395, -122.16578 37..."
1,ECR2068,d71492afe126cd61f629b17c3c056fcd,Bay Area 511 SamTrans Schedule,ECR,ECR,64,"LINESTRING (-122.46879 37.70620, -122.46897 37..."


In [13]:
# Get the stops for the target route, along with whether they're timepoints for the specific route
schedule_feed_entry = pd.read_sql_query(
    env.get_template("schedule_feed_entry.sql").render(
        {"FEED_KEY": feed_key}
    ),
    engine
)
assert schedule_feed_entry.index.size == 1
schedule_feed_valid_from_date = schedule_feed_entry.loc[0, "_valid_from"]

In [14]:
# Get timepoint stops and schedule rt stop times
schedule_rt_stop_times = pd.read_sql_query(
    env.get_template("schedule_rt_stop_times.sql").render(
        {
            "FEED_KEY": feed_key,
            "FEED_VALID_FROM": schedule_feed_valid_from_date,
            "SCHEDULE_BASE64_URL": schedule_feed_entry.loc[0, "base64_url"],
            "SHAPE_IDS": matching_shapes.shape_id.to_list(),
            **CONSTANT_MACROS
        }
    ),
    engine
)

# Get times between stops
schedule_rt_stop_times["prior_actual_departure_pacific"] = (
    schedule_rt_stop_times.groupby("trip_id")["actual_departure_pacific"].shift()
)
schedule_rt_stop_times["time_difference"] = (
    schedule_rt_stop_times["actual_arrival_pacific"] 
    - schedule_rt_stop_times["prior_actual_departure_pacific"]
)    
schedule_rt_stop_times["time_difference_cleaned"] = schedule_rt_stop_times["time_difference"].where(
    schedule_rt_stop_times["time_difference"] >= datetime.timedelta(0),
    pd.NaT
)
schedule_rt_stop_times["time_difference_seconds"] = (
    schedule_rt_stop_times["time_difference_cleaned"].dt.seconds
)
display(schedule_rt_stop_times.sort_values("shape_id"))

# calculate the variance metric
p80_time_difference = schedule_rt_stop_times.groupby(
    ["shape_id", "stop_sequence"]
)["time_difference_seconds"].quantile(0.8)
p80_time_difference.name = "p80_travel_time_seconds"
p20_time_difference = schedule_rt_stop_times.groupby(
    ["shape_id",  "stop_sequence"]
)["time_difference_seconds"].quantile(0.2)
p20_time_difference.name = "p20_travel_time_seconds"
time_difference_variance_metric = p80_time_difference / p20_time_difference
time_difference_variance_metric.name = "travel_time_variability"
n_trips_per_timepoint = schedule_rt_stop_times.groupby(["shape_id", "stop_sequence"])["trip_id"].count()
n_trips_per_timepoint.name = "n_rt_trips"
output_time_difference_variance = pd.concat(
    [time_difference_variance_metric, p80_time_difference, p20_time_difference, n_trips_per_timepoint],
    axis=1
).reset_index()
# messy - append to a list so we can concat later

,trip_id,trip_key,stop_id,shape_id,stop_sequence,actual_arrival_pacific,actual_departure_pacific,n_predictions,pt_geom,prior_actual_departure_pacific,time_difference,time_difference_cleaned,time_difference_seconds
4218,15273168-154-Blocks-Sunday-60,ff77075c8de6e901dbc998e851d3cc72,347082,ECR2068,82,2026-07-05 11:40:00,2026-07-05 11:40:00,90,POINT(-122.166052 37.443747),2026-07-05 11:37:32,0 days 00:02:28,0 days 00:02:28,148.0
1769,15273193-154-Blocks-Sunday-60,63d4a55473e4c6d3d2b2d43382cb71d2,335070,ECR2068,25,2026-07-05 16:26:36,2026-07-05 16:26:36,90,POINT(-122.41264 37.622643),2026-07-05 16:25:21,0 days 00:01:15,0 days 00:01:15,75.0
1770,15273193-154-Blocks-Sunday-60,63d4a55473e4c6d3d2b2d43382cb71d2,336627,ECR2068,30,2026-07-05 16:34:11,2026-07-05 16:34:11,90,POINT(-122.389438 37.600146),2026-07-05 16:26:36,0 days 00:07:35,0 days 00:07:35,455.0
1771,15273193-154-Blocks-Sunday-60,63d4a55473e4c6d3d2b2d43382cb71d2,336624,ECR2068,31,2026-07-05 16:38:00,2026-07-05 16:38:00,90,POINT(-122.386807 37.599656),2026-07-05 16:34:11,0 days 00:03:49,0 days 00:03:49,229.0
1772,15273193-154-Blocks-Sunday-60,63d4a55473e4c6d3d2b2d43382cb71d2,340008,ECR2068,36,2026-07-05 16:43:51,2026-07-05 16:43:51,90,POINT(-122.362851 37.587197),2026-07-05 16:38:00,0 days 00:05:51,0 days 00:05:51,351.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1729,15273209-154-Blocks-Sunday-60,61c6f071d69f750c051239a18f184723,345900,ECR2069,7,2026-07-05 04:58:53,2026-07-05 04:58:53,17,POINT(-122.209432 37.467725),2026-07-05 04:55:10,0 days 00:03:43,0 days 00:03:43,223.0
1730,15273209-154-Blocks-Sunday-60,61c6f071d69f750c051239a18f184723,344087,ECR2069,10,2026-07-05 05:02:15,2026-07-05 05:02:15,29,POINT(-122.223848 37.478081),2026-07-05 04:58:53,0 days 00:03:22,0 days 00:03:22,202.0
1731,15273209-154-Blocks-Sunday-60,61c6f071d69f750c051239a18f184723,344094,ECR2069,11,2026-07-05 05:03:02,2026-07-05 05:03:02,32,POINT(-122.227173 37.480493),2026-07-05 05:02:15,0 days 00:00:47,0 days 00:00:47,47.0
1676,15273238-154-Blocks-Sunday-60,61113fe018e002165bdf7647ab480978,333511,ECR2069,68,2026-07-05 14:31:00,2026-07-05 14:31:00,90,POINT(-122.465799 37.684826),2026-07-05 14:23:05,0 days 00:07:55,0 days 00:07:55,475.0


In [15]:
output_time_difference_variance

,shape_id,stop_sequence,travel_time_variability,p80_travel_time_seconds,p20_travel_time_seconds,n_rt_trips
0,ECR2068,1,NaN,NaN,NaN,23
1,ECR2068,2,1.495868,181.0,121.0,56
2,ECR2068,5,1.731481,187.0,108.0,59
3,ECR2068,8,1.431655,398.0,278.0,61
4,ECR2068,13,1.595541,501.0,314.0,61
...,...,...,...,...,...,...
62,ECR2069,62,2.210526,168.0,76.0,66
63,ECR2069,68,1.341525,535.0,398.8,66
64,ECR2069,71,1.833678,355.0,193.6,66
65,ECR2069,74,1.906207,276.4,145.0,65


In [16]:
# Create maps for each shape to show the stop variance metric for each segment

# Get a gdf of stops
timepoints = schedule_rt_stop_times[["shape_id", "stop_sequence", "stop_id", "pt_geom"]].drop_duplicates()
timepoints_gdf = gpd.GeoDataFrame(
    timepoints.drop(["pt_geom"], axis=1), 
    geometry=gpd.GeoSeries(timepoints["pt_geom"].map(shapely.wkt.loads), crs=4326)
)

for shape_id, shape_geom in matching_shapes[["shape_id", "geometry"]].itertuples(index=False):
    timepoints_for_shape = timepoints_gdf.loc[timepoints_gdf.shape_id == shape_id].copy()
    timepoints_for_shape["projected_distance"] = timepoints_for_shape.geometry.map(
        lambda point_geom: shape_geom.project(point_geom)
    )
    timepoints_for_shape["prior_projected_distance"] = timepoints_for_shape["projected_distance"].shift()
    timepoints_for_shape["segment_geometry"] = timepoints_for_shape[["projected_distance", "prior_projected_distance"]].apply(
        lambda row: (
            shapely.ops.substring(shape_geom, row["projected_distance"], row["prior_projected_distance"])
            if not row.isna().any()
            else None
        ),
        axis=1
    )
    segment_geometry_with_values = (
        timepoints_for_shape
        .set_geometry("segment_geometry")
        [
            ["shape_id", "stop_sequence", "stop_id", "segment_geometry"]
        ]
        .merge(
            output_time_difference_variance,
            on=["shape_id", "stop_sequence"],
            how="left"
        )
    )
    segment_geometry_with_values.set_crs(4326).explore(
        "travel_time_variability",
        tiles="CartoDB positron"
    ).save(f"{shape_id}.html")
